# Paper 1 — Golden Age Semantic Reconfiguration

**Phase 3 — Herrera edition resolution + dual temporal design**

Phase 2 established: 5,078 Navarro TEI poems, 53 authors, only one file with witness/edition dates, and 57 author–source groups. This phase resolves Herrera's H/P2 layers and separates **composition time** (primary) from **publication/circulation time** (secondary).

In [9]:
import sys,re,shutil,subprocess,unicodedata
from pathlib import Path
import pandas as pd
import xml.etree.ElementTree as ET

SOURCES={
"net":("https://github.com/lamusadecima/Network_for_Golden_Age_Spanish_Poetry.git","ef6b7b691f67abe60d9cfa85c274f0be8095dd9a"),
"nav":("https://github.com/bncolorado/CorpusSonetosSigloDeOro.git","092a5fe70a4065a4d84bfed288bffd3851348f9c"),
"dst":("https://github.com/lamusadecima/Digital-Stylistics-Applied-to-Golden-Age.git","0de990eac908897b5e931aeb5c496170ccf35bab")}
ROOT=Path("/content/gasr_sources"); ROOT.mkdir(exist_ok=True)
def clone(k,url,sha):
    d=ROOT/k
    if d.exists(): shutil.rmtree(d)
    subprocess.run(["git","clone","--quiet",url,str(d)],check=True)
    subprocess.run(["git","-C",str(d),"checkout","--quiet",sha],check=True)
    assert subprocess.check_output(["git","-C",str(d),"rev-parse","HEAD"],text=True).strip()==sha
    return d
P={k:clone(k,*v) for k,v in SOURCES.items()}; NET,NAV,DST=P["net"],P["nav"],P["dst"]
NS={"tei":"http://www.tei-c.org/ns/1.0"}
def norm(s):
    s=unicodedata.normalize("NFKD",str(s))
    s="".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"[^a-z0-9]","",s.lower())
def blocks(s):
    out=[]; cur=[]
    for x in s.splitlines():
        if x.strip(): cur.append(x.strip())
        elif cur: out.append(cur); cur=[]
    if cur: out.append(cur)
    return out
def fb(path,label):
    z=[]
    for i,b in enumerate(blocks(path.read_text(encoding="utf-8",errors="replace")),1):
        t="\n".join(b); z.append(dict(layer=label,id=i,n_lines=len(b),text=t,signature=norm(t)))
    return pd.DataFrame(z)
def et(el): return "" if el is None else " ".join(" ".join(el.itertext()).split())
print("Sources pinned; Python",sys.version.split()[0])

Sources pinned; Python 3.13.15


## 1. Resolve H and P2 empirically

Hernández-Lorenzo defines **H** as the sonnets of *Algunas obras* (1582), and **P2** as the sonnets unique to the posthumous *Versos de Fernando de Herrera* (1619), excluding poems already published in H. The network corpus contains `AN_SonetosP2.txt`; we verify its identity against the companion repository's explicit `P2.txt`.

In [10]:
an=fb(NET/"corpus"/"AN_SonetosP2.txt","network_AN")
hh=fb(NET/"corpus"/"Herrera_Sonetos.txt","network_Herrera")
pa=fb(NET/"corpus"/"Pacheco_Sonetos.txt","Pacheco")
U=DST/"corpus"/"untagged_corpus"
H=fb(U/"H.txt","H_1582"); P2=fb(U/"P2.txt","P2_1619")
display(pd.DataFrame([{"layer":x.layer.iloc[0],"blocks":len(x),"blocks_14":int((x.n_lines==14).sum())} for x in [an,hh,pa,H,P2]]))
def ov(a,b):
    A,B=set(a.signature),set(b.signature); sh=len(A&B)
    return dict(A=a.layer.iloc[0],B=b.layer.iloc[0],unique_A=len(A),unique_B=len(B),
                exact_shared=sh,coverage_A=sh/len(A),coverage_B=sh/len(B))
edition_overlap=pd.DataFrame([ov(an,P2),ov(an,H),ov(hh,H),ov(hh,P2),ov(H,P2)])
display(edition_overlap)

,layer,blocks,blocks_14
0,network_AN,182,182
1,network_Herrera,179,178
2,Pacheco,22,22
3,H_1582,78,78
4,P2_1619,200,172


,A,B,unique_A,unique_B,exact_shared,coverage_A,coverage_B
0,network_AN,P2_1619,182,199,165,0.906593,0.829146
1,network_AN,H_1582,182,78,1,0.005495,0.012821
2,network_Herrera,H_1582,179,78,78,0.435754,1.000000
3,network_Herrera,P2_1619,179,199,0,0.000000,0.000000
4,H_1582,P2_1619,78,199,0,0.000000,0.000000


## 2. Tag Navarro Herrera poems by exact edition membership

Exact matches to H or P2 can receive secure **circulation** labels (1582 or 1619). These years are **not composition dates**.

In [11]:
rows=[]
for p in sorted(NAV.rglob("*.xml")):
    r=ET.parse(p).getroot()
    ls=[et(x) for x in r.findall(".//tei:l",NS)]; ls=[x for x in ls if x]
    text="\n".join(ls)
    rows.append(dict(n_id=str(p.relative_to(NAV)).replace("/","::"),author_dir=p.parent.name,
        title=et(r.find(".//tei:body/tei:head/tei:title",NS)),text_tei=text,n_lines=len(ls),
        source_bibl=et(r.find(".//tei:sourceDesc/tei:bibl",NS)),signature=norm(text)))
n=pd.DataFrame(rows)
herr=n[n.author_dir.eq("FernandoDeHerrera")].copy()
Hs,P2s=set(H.signature),set(P2.signature)
herr["edition_exact"]=herr.signature.map(lambda s:"H_1582" if s in Hs else ("P2_1619" if s in P2s else "unclassified"))
herr["circulation_year"]=herr.edition_exact.map({"H_1582":1582,"P2_1619":1619})
print("Navarro Herrera poems:",len(herr))
display(herr.edition_exact.value_counts().rename_axis("edition").reset_index(name="poems"))

Navarro Herrera poems: 320


,edition,poems
0,unclassified,231
1,P2_1619,63
2,H_1582,26


## 3. Temporal design

**Composition axis (primary):** exact scholarly date or bounded composition interval.  
**Circulation axis (secondary robustness):** first publication/collection date.

For Herrera, 1582 and 1619 belong to the circulation axis. Assigning P2 to composition year 1619 would be historically invalid because Herrera died in 1597. The gap between composition-time and circulation-time analyses can become a substantive robustness test rather than a nuisance.

In [12]:
source_groups=(n.groupby(["author_dir","source_bibl"],dropna=False).agg(poems=("n_id","count")).reset_index())
central={"GarcilasoDeLaVega","JuanBoscan","FernandoDeHerrera","PedroEspinosa","JuanDeArguijo",
"JuanDeJauregui","LuisCarrilloYSotomayor","Cervantes","Gongora","LopeDeVega_1","LopeDeVega_2","Quevedo"}
source_groups["priority"]=source_groups.apply(lambda r:"A" if r.author_dir in central else ("B" if r.poems>=100 else "C"),axis=1)
source_groups["needed"]="critical edition/scholarly chronology: composition year or bounded interval"
source_groups=source_groups.sort_values(["priority","poems"],ascending=[True,False])
print("Author + source groups:",len(source_groups),"| authors:",n.author_dir.nunique())
display(source_groups.head(35))
schema=pd.DataFrame([
["composition_exact","composition","A","primary"],
["composition_interval","composition","A/B","primary"],
["first_publication","circulation","B","secondary"],
["witness_or_edition","bibliographic","C","never composition"],
["author_activity_interval","composition","D","uncertainty fallback"]],
columns=["date_type","axis","confidence","role"])
display(schema)

Author + source groups: 57 | authors: 53


,author_dir,source_bibl,poems,priority,needed
40,LopeDeVega_1,Sonetos de Lope de Vega . Biblioteca Virtual M...,699,A,critical edition/scholarly chronology: composi...
41,LopeDeVega_2,Sonetos de Lope de Vega . Biblioteca Virtual M...,647,A,critical edition/scholarly chronology: composi...
53,Quevedo,Sonetos de Quevedo . Biblioteca Virtual Miguel...,517,A,critical edition/scholarly chronology: composi...
17,FernandoDeHerrera,Sonetos de Fernando de Herrera . Biblioteca Vi...,320,A,critical edition/scholarly chronology: composi...
26,Gongora,Sonetos de Gongora . Biblioteca Virtual Miguel...,113,A,critical edition/scholarly chronology: composi...
32,JuanBoscan,Sonetos de Juan Boscan . Biblioteca Virtual Mi...,100,A,critical edition/scholarly chronology: composi...
9,Cervantes,Sonetos de Cervantes . Biblioteca Virtual Migu...,77,A,critical edition/scholarly chronology: composi...
34,JuanDeArguijo,"Sonetos de Arguijo, Juan de . Biblioteca Virtu...",70,A,critical edition/scholarly chronology: composi...
24,GarcilasoDeLaVega,Sonetos de Garcilaso de La Vega . Biblioteca V...,38,A,critical edition/scholarly chronology: composi...
35,JuanDeJauregui,"Sonetos de Jauregui, Juan de . Biblioteca Virt...",23,A,critical edition/scholarly chronology: composi...


,date_type,axis,confidence,role
0,composition_exact,composition,A,primary
1,composition_interval,composition,A/B,primary
2,first_publication,circulation,B,secondary
3,witness_or_edition,bibliographic,C,never composition
4,author_activity_interval,composition,D,uncertainty fallback


In [13]:
OUT=Path("/content/gasr_phase3_outputs"); OUT.mkdir(exist_ok=True)
edition_overlap.to_csv(OUT/"herrera_edition_overlap.csv",index=False)
herr[["n_id","title","edition_exact","circulation_year"]].to_csv(OUT/"herrera_temporal_evidence.csv",index=False)
source_groups.to_csv(OUT/"temporal_source_worklist.csv",index=False)
schema.to_csv(OUT/"temporal_schema.csv",index=False)
print("PHASE 3 CHECKPOINT")
print("Save this executed notebook to GitHub.")
print("Do NOT build semantic networks yet.")
print("Next: interpret H/P2 identity and begin scholarly dating of priority-A source groups.")

PHASE 3 CHECKPOINT
Save this executed notebook to GitHub.
Do NOT build semantic networks yet.
Next: interpret H/P2 identity and begin scholarly dating of priority-A source groups.
